# Atlas del Carbono: Evolución Histórica de Emisiones
Este análisis documenta paso a paso el proceso de estructuración estadística de la historia de las emisiones globales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.1)

### 1. Ingesta y Purgado de Datos (Estándar ISO)
Filtramos la base global. Descartamos macro-regiones y bloques económicos (ej. 'High-income countries') utilizando el estándar internacional de código ISO para mantener únicamente países soberanos.

In [ ]:
url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
raw_df = pd.read_csv(url)

# Llenar ceros numéricos pero preservar nulos en texto
df = raw_df.copy()
for c in ['gdp', 'population', 'co2']:
    if c in df.columns: df[c] = df[c].fillna(0)

# FORMA CORRECTA DE FILTRAR PAÍSES PUROS EN OWID
# Mantenemos solo filas donde iso_code existe y no es un prefijo de OWID (regiones/continentes)
df_countries = df[df['iso_code'].notna() & ~df['iso_code'].str.startswith('OWID', na=False)].copy()

# Reservamos solo los continentes clásicos para el análisis regional
continents = ['World', 'Asia', 'Oceania', 'Europe', 'Africa', 'North America', 'South America']
df_continents = df[df['country'].isin(continents)].copy()

print(f"Países reales filtrados: {df_countries['country'].nunique()}")
display(df_countries[['country', 'iso_code', 'year', 'co2']].tail(3))

### 2. Estandarización Per Cápita
El impacto individual se calcula a partir del volumen demográfico real.

In [ ]:
df_countries['gdp_per_capita'] = np.where(df_countries['population'] > 0, df_countries['gdp'] / df_countries['population'], 0)
df_countries['co2_per_capita'] = np.where(df_countries['population'] > 0, df_countries['co2'] / (df_countries['population']/1e6), 0)

display(df_countries[['country', 'year', 'gdp_per_capita', 'co2_per_capita']].describe().round(2))

### 3. Top 10 Infractores Históricos (Países Reales)
Evaluamos qué Estados soberanos han emitido más dióxido de carbono en términos absolutos desde el inicio de la revolución industrial.

In [ ]:
historico = df_countries.groupby('country')['co2'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 4))
sns.barplot(x=historico.values, y=historico.index, palette='Reds_r')
plt.title('La Huella Real: Emisiones Acumuladas en la Historia')
plt.xlabel('Millones de Toneladas CO2')
plt.ylabel('')
plt.show()

display(pd.DataFrame(historico).reset_index().rename(columns={'co2': 'Acumulado_Historico(Mtons)'}))

### 4. Transición de Hegemonía Energética
Observamos el declive europeo y norteamericano frente a la inflexión asiática a principios del nuevo milenio.

In [ ]:
df_trend = df_continents[(df_continents['year'] >= 1950) & (df_continents['year'] <= 2024)]

plt.figure(figsize=(10, 5))
sns.lineplot(data=df_trend[df_trend.country.isin(['Asia', 'Europe', 'North America', 'Africa'])], 
             x='year', y='co2', hue='country', linewidth=2)
plt.title('Transición Energética Mundial (1950 - 2024)')
plt.axvline(2000, color='red', linestyle='--', alpha=0.5, label='Inflexión Año 2000')
plt.legend()
plt.ylabel('Emisiones (Millones Toneladas)')
plt.show()

pivot_table = df_trend[df_trend['year'].isin([1950, 1970, 1990, 2010, 2024])].pivot_table(index='year', columns='country', values='co2').round(2)
display(pivot_table[['Asia', 'Europe', 'North America', 'Africa']])

### 5. Estado Actual: Mayores Emisores Contemporáneos (2024)
En nuestra última radiografía anual disponible, los 5 principales emisores globales.

In [ ]:
recent_year = df_countries['year'].max()
df_latest = df_countries[df_countries['year'] == recent_year]
top_5_actual = df_latest.sort_values('co2', ascending=False).head(5)

plt.figure(figsize=(10, 4))
sns.barplot(data=top_5_actual, x='co2', y='country', palette='Oranges_r')
plt.title(f'Top 5 Emisores Actuales ({recent_year})')
plt.xlabel('Emisiones de C02 (M Toneladas)')
plt.ylabel('')
plt.show()

display(top_5_actual[['country', 'iso_code', 'co2', 'co2_per_capita']])

### 6. Desigualdad Contemporánea (Clúster Población vs CO2)
Visualizamos la dispersión logarítmica para denotar cómo la economía está atada al carbono.

In [ ]:
df_modern = df_latest[(df_latest['gdp_per_capita'] > 100) & (df_latest['co2'] > 0)]

plt.figure(figsize=(9,5))
plt.xscale('log')
plt.yscale('log')
sns.regplot(data=df_modern, x='gdp_per_capita', y='co2', 
                 scatter_kws={'alpha':0.4, 'color':'#10b981'}, 
                 line_kws={'color':'#ea580c', 'linewidth':2})
plt.title(f'Desigualdad Global ({recent_year}): Log PIB vs Log CO2')
plt.xlabel('PIB per Cápita (USD)')
plt.ylabel('Emisiones CO2 (M Toneladas)')
plt.show()

display(df_modern[['country', 'gdp_per_capita', 'co2']].sort_values('gdp_per_capita', ascending=False).head(5))

### 7. Análisis Soberano (Ej. Colombia)
Inspección individual para auditoría general.

In [ ]:
df_colombia = df_countries[(df_countries['country'].str.contains('Colombia')) & (df_countries['year'] >= 1950)]

plt.figure(figsize=(10, 4))
sns.lineplot(data=df_colombia, x='year', y='co2', color='#1d4ed8', linewidth=2.5)
plt.title('Trayectoria de Emisiones: Colombia')
plt.fill_between(df_colombia.year, df_colombia.co2, color='#1d4ed8', alpha=0.1)
plt.ylabel('M Toneladas')
plt.show()

display(df_colombia[['year', 'co2']].tail(3))

### 8. Exportación de Resultados
Generamos un duplicado en formato HTML/PDF guardado automáticamente en tu carpeta Downloads.

In [ ]:
import os
from pathlib import Path

downloads_path = str(Path.home() / 'Downloads' / '04_co2_analysis_owid.html')

# Guardamos estado (requiere que el usuario haya guardado el notebook (Cmd+S) para tener la versión final)
print(f"Exportando reporte al directorio: {downloads_path}")
code = os.system(f'jupyter nbconvert --to html 04_co2_analysis_owid.ipynb --output {downloads_path}')

if code == 0:
    print("¡Exportación HTML generada con éxito en tu carpeta Downloads! Puedes abrirla en el navegador y darle imprimir (Export to PDF).")
else:
    print("Nota: Si no funcionó nbconvert nativo desde aquí, usa en el menú superior de VSCode o Jupyter la opción: File -> Export Notebook As -> PDF/HTML.")